In [ ]:
# Cell: Install Gradio
!pip install gradio opencv-python-headless -q

In [ ]:
# Cell: DeepGuard — GradCAM + Gradio Frontend
import gradio as gr
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image
import matplotlib
matplotlib.use("Agg")

# ── GradCAM ───────────────────────────────────────────────────────────────────
def get_gradcam_heatmap(mdl, img_array, last_conv_layer_name="top_conv"):
    grad_model = tf.keras.models.Model(
        inputs=mdl.inputs,
        outputs=[mdl.get_layer(last_conv_layer_name).output, mdl.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]
    grads        = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap      = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap).numpy()
    heatmap      = np.maximum(heatmap, 0)
    if heatmap.max() != 0:
        heatmap /= heatmap.max()
    return heatmap

def overlay_gradcam(original_img_pil, heatmap, alpha=0.45):
    img  = np.array(original_img_pil.resize((300, 300)))
    heat = cv2.resize(heatmap, (300, 300))
    heat = np.uint8(255 * heat)
    heat = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    blend = (img * (1 - alpha) + heat * alpha).astype(np.uint8)
    return Image.fromarray(blend)

# ── Predict ───────────────────────────────────────────────────────────────────
def predict_with_gradcam(pil_img, threshold=0.45):
    if pil_img is None:
        return None, "<p style='color:#555'>No image uploaded.</p>"

    img_resized = pil_img.resize((300, 300))
    x = keras_image.img_to_array(img_resized)
    x = preprocess_input(x)
    x = np.expand_dims(x, axis=0)

    prob  = float(model.predict(x, verbose=0)[0][0])
    label = "REAL" if prob >= threshold else "FAKE"

    try:
        heatmap     = get_gradcam_heatmap(model, x)
        gradcam_img = overlay_gradcam(img_resized, heatmap)
    except Exception as e:
        gradcam_img = img_resized

    real_conf = prob * 100
    fake_conf = (1 - prob) * 100
    color     = "#00e5a0" if label == "REAL" else "#ff4d6d"
    icon      = "✓" if label == "REAL" else "✗"

    html = f"""
    <div style="font-family:'Space Mono',monospace;padding:16px 0;">
      <div style="font-size:48px;font-weight:900;color:{color};letter-spacing:-2px;line-height:1">
        {icon} {label}
      </div>
      <div style="margin-top:20px;font-size:11px;color:#666;letter-spacing:3px;text-transform:uppercase;margin-bottom:10px">
        Confidence Breakdown
      </div>
      <div style="margin-bottom:8px">
        <div style="display:flex;justify-content:space-between;font-size:11px;color:#888;margin-bottom:4px">
          <span style="letter-spacing:2px">REAL</span><span>{real_conf:.1f}%</span>
        </div>
        <div style="background:#1a1a2e;border-radius:3px;height:8px;overflow:hidden">
          <div style="width:{real_conf:.1f}%;background:#00e5a0;height:100%;border-radius:3px"></div>
        </div>
      </div>
      <div style="margin-bottom:8px">
        <div style="display:flex;justify-content:space-between;font-size:11px;color:#888;margin-bottom:4px">
          <span style="letter-spacing:2px">FAKE</span><span>{fake_conf:.1f}%</span>
        </div>
        <div style="background:#1a1a2e;border-radius:3px;height:8px;overflow:hidden">
          <div style="width:{fake_conf:.1f}%;background:#ff4d6d;height:100%;border-radius:3px"></div>
        </div>
      </div>
      <div style="margin-top:14px;padding-top:12px;border-top:1px solid #1e1e2e;font-size:10px;color:#444;letter-spacing:1px">
        Raw score: {prob:.6f} &nbsp;·&nbsp; Threshold: {threshold} &nbsp;·&nbsp; Model: EfficientNetB3
      </div>
    </div>
    """
    return gradcam_img, html

# ── CSS ───────────────────────────────────────────────────────────────────────
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Bebas+Neue&display=swap');

:root {
  --bg:      #0a0a0f;
  --surface: #0f0f18;
  --border:  #1e1e2e;
  --green:   #00e5a0;
  --red:     #ff4d6d;
  --muted:   #44445a;
  --text:    #d0d0e8;
}

body, .gradio-container {
  background: var(--bg) !important;
  color: var(--text) !important;
  font-family: 'Space Mono', monospace !important;
}

.gradio-container {
  max-width: 1100px !important;
  margin: 0 auto !important;
}

/* Scanlines */
.gradio-container::after {
  content: '';
  position: fixed;
  inset: 0;
  background: repeating-linear-gradient(
    0deg, transparent, transparent 3px,
    rgba(0,229,160,0.012) 3px, rgba(0,229,160,0.012) 4px
  );
  pointer-events: none;
  z-index: 9999;
}

/* Header */
#header-html {
  text-align: center;
  padding: 48px 24px 36px;
  border-bottom: 1px solid var(--border);
  margin-bottom: 28px;
}

/* Cards */
.card {
  background: var(--surface) !important;
  border: 1px solid var(--border) !important;
  border-radius: 3px !important;
  padding: 24px !important;
}

/* Upload */
.upload-zone {
  border: 1px dashed rgba(0,229,160,0.4) !important;
  background: rgba(0,229,160,0.02) !important;
  border-radius: 3px !important;
  min-height: 260px !important;
}
.upload-zone:hover {
  background: rgba(0,229,160,0.05) !important;
  border-color: var(--green) !important;
}

/* Buttons */
button.lg.primary {
  background: var(--green) !important;
  color: #000 !important;
  font-family: 'Space Mono', monospace !important;
  font-weight: 700 !important;
  font-size: 12px !important;
  letter-spacing: 4px !important;
  text-transform: uppercase !important;
  border: none !important;
  border-radius: 2px !important;
  transition: all 0.2s !important;
}
button.lg.primary:hover {
  background: #00ffb3 !important;
  box-shadow: 0 0 32px rgba(0,229,160,0.3) !important;
}
button.lg.secondary {
  background: transparent !important;
  color: var(--muted) !important;
  font-family: 'Space Mono', monospace !important;
  font-size: 11px !important;
  letter-spacing: 2px !important;
  border: 1px solid var(--border) !important;
  border-radius: 2px !important;
  transition: all 0.2s !important;
}
button.lg.secondary:hover {
  color: var(--text) !important;
  border-color: #444 !important;
}

/* Slider */
input[type=range] { accent-color: var(--green) !important; }

/* Labels */
label span, .block > label > span {
  font-family: 'Space Mono', monospace !important;
  font-size: 10px !important;
  letter-spacing: 3px !important;
  text-transform: uppercase !important;
  color: var(--muted) !important;
}

/* Output image */
img { border: 1px solid var(--border); border-radius: 2px; }

/* Scrollbar */
::-webkit-scrollbar { width: 3px; }
::-webkit-scrollbar-thumb { background: var(--border); }
"""

HEADER_HTML = """
<div id="header-html">
  <div style="font-size:10px;letter-spacing:5px;color:#333;margin-bottom:16px;text-transform:uppercase">
    ── Neural Vision Lab ──
  </div>
  <div style="font-family:'Bebas Neue',sans-serif;font-size:clamp(56px,10vw,100px);
              color:#d0d0e8;letter-spacing:8px;line-height:0.85;margin:0">
    DEEP<span style="color:#00e5a0">GUARD</span>
  </div>
  <div style="margin-top:14px;font-size:10px;letter-spacing:5px;color:#44445a;text-transform:uppercase">
    Deepfake Detection · GradCAM Explainability · EfficientNetB3
  </div>
  <div style="margin-top:20px;display:flex;gap:10px;justify-content:center;flex-wrap:wrap">
    <span style="font-size:10px;letter-spacing:2px;color:#333;border:1px solid #1e1e2e;padding:4px 12px">
      ACC <b style="color:#00e5a0">90%</b>
    </span>
    <span style="font-size:10px;letter-spacing:2px;color:#333;border:1px solid #1e1e2e;padding:4px 12px">
      F1 <b style="color:#00e5a0">0.90</b>
    </span>
    <span style="font-size:10px;letter-spacing:2px;color:#333;border:1px solid #1e1e2e;padding:4px 12px">
      CLASSES <b style="color:#00e5a0">Real / Fake</b>
    </span>
    <span style="font-size:10px;letter-spacing:2px;color:#333;border:1px solid #1e1e2e;padding:4px 12px">
      MTCNN <b style="color:#00e5a0">Face Crop</b>
    </span>
  </div>
</div>
"""

HOW_IT_WORKS = """
<div style="font-family:'Space Mono',monospace;font-size:11px;color:#333;
            line-height:2.2;padding:8px 0;border-top:1px solid #1e1e2e;margin-top:8px">
  <div style="color:#00e5a0;letter-spacing:4px;font-size:9px;margin-bottom:10px">HOW IT WORKS</div>
  <span style="color:#44445a">01</span> Upload a face image (raw or cropped)<br>
  <span style="color:#44445a">02</span> EfficientNetB3 runs inference<br>
  <span style="color:#44445a">03</span> GradCAM maps attention regions<br>
  <span style="color:#44445a">04</span> Result + confidence returned<br>
  <div style="margin-top:14px;color:#222;font-size:9px;letter-spacing:2px">
    ── Warm regions = high model attention ──
  </div>
</div>
"""

FOOTER_HTML = """
<div style="text-align:center;padding:24px;border-top:1px solid #1e1e2e;
            margin-top:24px;font-family:'Space Mono',monospace;
            font-size:9px;letter-spacing:3px;color:#2a2a3a">
  DEEPGUARD · EfficientNetB3 + GradCAM · TensorFlow + Gradio
  <br><span style="color:#1e1e2e">⚠ Research & educational use only</span>
</div>
"""

AWAIT_HTML = """
<div style="font-family:'Space Mono',monospace;color:#1e1e2e;
            font-size:12px;letter-spacing:3px;padding:32px 0;text-align:center">
  ── AWAITING INPUT ──
</div>
"""

# ── Layout ────────────────────────────────────────────────────────────────────
with gr.Blocks(css=CSS, title="DeepGuard · Deepfake Detector") as demo:

    gr.HTML(HEADER_HTML)

    with gr.Row(equal_height=False):

        # LEFT
        with gr.Column(scale=1, elem_classes=["card"]):
            img_input = gr.Image(
                type="pil",
                label="Upload Face Image",
                elem_classes=["upload-zone"],
                height=280
            )
            threshold = gr.Slider(
                minimum=0.2, maximum=0.8, value=0.45, step=0.05,
                label="Detection Threshold — lower = stricter on fakes"
            )
            with gr.Row():
                run_btn   = gr.Button("⬡  ANALYZE IMAGE", variant="primary", size="lg")
                clear_btn = gr.Button("CLEAR", variant="secondary", size="lg")
            gr.HTML(HOW_IT_WORKS)

        # RIGHT
        with gr.Column(scale=1, elem_classes=["card"]):
            gradcam_out = gr.Image(
                type="pil",
                label="GradCAM · Attention Heatmap",
                height=300
            )
            result_html = gr.HTML(value=AWAIT_HTML)

    gr.HTML(FOOTER_HTML)

    run_btn.click(
        fn=predict_with_gradcam,
        inputs=[img_input, threshold],
        outputs=[gradcam_out, result_html]
    )
    clear_btn.click(
        fn=lambda: (None, None, AWAIT_HTML),
        inputs=[],
        outputs=[img_input, gradcam_out, result_html]
    )

demo.launch(share=True, debug=False)
